# International Brain Laboratory Brain-Wide Map

Neuropixels recordings from mice making visual decisions. On each trial a grating appears on the left or right of a screen, and the mouse turns a wheel to bring it to the center for a reward. The International Brain Laboratory recorded across 12 labs and nearly every brain region. Each session here has:

- the raw voltage of one or two probes, 384 channels each, with every channel's position and brain region;
- IBL's spike sorting of each probe, as published in 2022 and as re-sorted in 2024;
- the trials and the mouse's behavior: wheel, pupil, licks and pose.

The data is CC BY 4.0; cite the Brain-Wide Map paper ([doi:10.1038/s41586-025-09235-0](https://doi.org/10.1038/s41586-025-09235-0)) and DANDI 000409.

The repo is `earthmover-biology/ibl-brain-wide-map`. You need an Arraylake login with read access (`al auth login`), and zarr and xarray. Sessions are named as on DANDI, `sub-<subject>/ses-<session id>`.

In [ ]:
import numpy as np
import xarray as xr
import zarr
from arraylake import Client

repo = Client().get_repo("earthmover-biology/ibl-brain-wide-map")
store = repo.readonly_session("main").store
root = zarr.open_group(store, mode="r")

print(root.attrs["title"], "|", root.attrs["license"])
sessions = [f"{s}/{e}" for s, subject in root.groups() for e, _ in subject.groups()]
for path in sessions:
    print(f"{path}: {root[path].attrs['description']}")

## A session's recordings

`raw` holds the session's raw recordings and `processed` its trials and behavior, each as the session's NWB file lays them out. zarr reads them directly. To open them with pynwb, hdmf-zarr has to be able to open a group inside a store, which it cannot yet. The electrodes table lists both probes' channels, and a series' `electrodes` array gives the table rows of its channels.

In [ ]:
session = sessions[0]
raw = root[f"{session}/raw"]
print(raw.attrs["neurodata_type"], raw.attrs["nwb_version"], "|", raw["session_start_time"][()], "|", raw["general/subject/subject_id"][()])
print("acquisition:", sorted(raw["acquisition"].group_keys()))

electrodes = xr.open_zarr(store, group=f"{session}/raw/general/extracellular_ephys/electrodes", consolidated=False)
series = raw["acquisition/ElectricalSeriesProbe00AP"]
rows = series["electrodes"][:]
print(series["data"].shape, series["data"].dtype, "conversion", series["data"].attrs["conversion"], series["data"].attrs["unit"])
print(electrodes[["probe_name", "rel_x", "rel_y", "location"]].isel(electrodes=rows[:5]).to_dataframe())

## The same series in xarray

Every array also has dimension names, so an ElectricalSeries opens as an xarray Dataset. One chunk is 13,020 samples of all 384 channels. The voltages carry NWB's conversion as a CF `scale_factor`, which xarray applies in float64 by default, making one AP band a 332 GB variable; opening with `mask_and_scale=False` and scaling in float32 halves that, and float32 holds every int16 sample exactly.

In [ ]:
ap = xr.open_zarr(store, group=f"{session}/raw/acquisition/ElectricalSeriesProbe00AP",
                  consolidated=False, mask_and_scale=False)
volts = ap["data"].astype("float32") * np.float32(ap["data"].attrs["scale_factor"])
rate = float(ap.attrs["spikeglx_meta"]["imSampRate"])
second = volts.isel(time=slice(int(1000 * rate), int(1001 * rate))).compute()
print(f"{volts.dtype}, {volts.nbytes / 1e9:.0f} GB for the whole band")
print(f"nominal rate {rate} Hz; one second starts at t = {float(second.timestamps[0]):.4f} s")
print("RMS per channel, microvolts:", np.round(1e6 * np.sqrt((second ** 2).mean("time")).values[:8], 1))

## The two spike sorts

IBL sorted the whole dataset in 2022 and again in 2024. Each sort is a tagged version of the store; `main` holds the 2024 sort.

In [ ]:
for tag in ["sort-2022", "sort-2024"]:
    tagged = zarr.open_group(repo.readonly_session(tag=tag).store, mode="r")
    sorting = tagged[f"{session}/alf/probe00/pykilosort"]
    spikes = sorting["spikes/times"]
    clusters = sorting["clusters/depths"].shape[0]
    print(f"{tag}: revision {sorting.attrs['ibl_revision']}, {spikes.shape[0]:,} spikes in {clusters} clusters")

## Behavior

The processed NWB file holds the trials table. Its columns share the `trials` axis, so the table opens as an xarray Dataset.

In [ ]:
trials = xr.open_zarr(store, group=f"{session}/processed/intervals/trials", consolidated=False)
print(trials.sizes["trials"], "trials")
print(trials[["gabor_stimulus_contrast", "is_mouse_rewarded", "start_time"]].to_dataframe().head())